In [ ]:
!pip install statsmodels==0.14.0 # avoid errors

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import boto3
import sagemaker
from statsmodels.tsa.arima.model import ARIMA
from sagemaker.session import Session
from sagemaker.sklearn.estimator import SKLearn 
from sagemaker.sklearn.model import SKLearnModel 

import joblib
from time import strftime, gmtime 

In [ ]:
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
sm_boto3 = boto3.client('sagemaker')

In [ ]:
#s3_client = boto_session.client('s3') 
daily_energy = pd.read_csv('daily_energy_train.csv')

In [ ]:
def preprocess_exog(df: pd.DataFrame):
    df['Datetime'] = pd.to_datetime(df['Datetime'], format='%Y-%m-%d')
    df.sort_values(by='Datetime', inplace=True)
    df.reset_index(inplace=True, drop=True)
    df = df.resample('D', on='Datetime').sum()
    df.index.freq = 'D'
    
    exog3 = pd.DataFrame({'Datetime': df.index})
    exog3 = exog3.set_index(pd.DatetimeIndex(exog3['Datetime'], freq='D'))
    exog3['sin365'] = np.sin(2 * np.pi * exog3.index.dayofyear / 365.25)
    exog3['cos365'] = np.cos(2 * np.pi * exog3.index.dayofyear / 365.25)
    exog3['sin365_2'] = np.sin(4 * np.pi * exog3.index.dayofyear / 365.25)
    exog3['cos365_2'] = np.cos(4 * np.pi * exog3.index.dayofyear / 365.25)
    exog3 = exog3.drop(columns=['Datetime'])

    return df, exog3

In [ ]:
def preprocess(df: pd.DataFrame):
    df['Datetime'] = pd.to_datetime(df['Datetime'], format='%Y-%m-%d')
    df.sort_values(by='Datetime', inplace=True)
    df.reset_index(inplace=True, drop=True)
    df = df.resample('D', on='Datetime').sum()
    df.index.freq = 'D'
    return df

In [ ]:
train_df = preprocess(pd.read_csv('daily_energy_train.csv'))
test_df = preprocess(pd.read_csv('daily_energy_test.csv'))

In [ ]:
train_df.tail(5)

In [ ]:
test_df.head(5)

In [ ]:
model = ARIMA( 
        train_df,
        order=(3,0,0),
        seasonal_order=(0,1,1,7),
)
model = model.fit()
joblib.dump(model, "model.joblib")

In [ ]:
def forecast_accuracy(forecast, actual):
    forecast, actual = np.array(forecast), np.array(actual)
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))*100  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return({'mape(%)':mape, 'mae': mae, 'rmse':rmse})

In [ ]:
predictions = model.forecast(len(test_df))
forecast_accuracy(predictions, test_df)

In [ ]:
%%writefile train.py

import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import joblib
import argparse
import os
from sagemaker.serializers import NumpySerializer, CSVSerializer



#def preprocess(df: pd.DataFrame):
#    df['Datetime'] = pd.to_datetime(df['Datetime'], format='%Y-%m-%d')
#    df.sort_values(by='Datetime', inplace=True)
#    df.reset_index(inplace=True, drop=True)
#    df = df.resample('D', on='Datetime').sum()
#    df.index.freq = 'D'
#    return df
#

def forecast_accuracy(forecast, actual):
    #forecast, actual = np.asarray(forecast), np.asarray(actual)
    forecast, actual = np.array(forecast), np.array(actual)
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))*100  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return({'mape(%)':mape, 'mae': mae, 'rmse':rmse})


def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))
    return clf



def input_fn(input_data, content_type):
    """ 
    param input_data : raw data passed by user/reques
    param contenty_type: type of data passed
    """
    print("input_fn-----------------------")
    pritn('content_type', content_type)
    if content_type == "text/csv":
        df = CSVSerializer().serialize(input_data)
        return np.array(df['PJME'])

    elif content_type == "application/x-npy":
        # Use NumpySerializer for application/x-npy content type
        serializer = NumpySerializer()
        return serializer.serialize(input_data)

    else:
        # If the content type is not as expected, raise an exception
        raise ValueError(f"Content type {content_type} is not supported")

def predict_fn(input_data, model):
    """
    param input_data: returned by the input_fn
    param model: returned by the model_fn
    returns: predictions 
    """
    predictions = model_fit.forecast(len(input_data))
    predictions = {"predictions": predictions.tolist()}
    print("predict_fn-----------------------")
    return np.array(predictions)


def output_fn(predictions, accept):
    """
    param: predictions returned from predict_fn
    returns: predictions correctly transformed 
    Post-processing function for model predictions. It gets executed after the predict_fn.
    """
    if accept != "application/x-npy":
        raise ValueError(f"Accept type {accept} is not supported")
    print("output_fn-----------------------")
    #worker.Response(NumpyDeserializer().deserialize(predictions), accept, mimetype=accept)
    return NumpyDeserializer().deserialize(predictions)


def _list_of_ints(arg):
    # for argparser
    return list(map(int, arg.split(',')))

if __name__=='__main__':
    parser = argparse.ArgumentParser()

    # where to write trained model artifacts
    parser.add_argument(
        "--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR")
    )

    # Train and Test Data
    parser.add_argument(
        "--train", type=str, default='daily_energy_clean_train.csv'
    )
    parser.add_argument(
        "--test", type=str, default='daily_energy_clean_test.csv'
        #default=os.path.join(
        #    os.environ.get("SM_CHANNEL_TRAIN"), "daily_energy_test.csv")
    )
    
    # Hyperparameters
    parser.add_argument(
        "--order", type=_list_of_ints, default=(3,0,0)
    )
    parser.add_argument(
        "--seasonal_order", type=_list_of_ints, default=(0,1,1,7)
    )

    args, _ = parser.parse_known_args()

    print("\n[INFO] Reading data\n")
    path = os.getcwd()
    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)
    
    print("\n[INFO] Training data with ARIMA")
    model = ARIMA(
        train_df, 
        order=args.order,
        seasonal_order=args.seasonal_order
    )
    model = model.fit()

    model_path = os.path.join(args.model_dir, "model.joblib")
    print("\n[INFO] Saving model at", model_path)
    joblib.dump(model, model_path)


    print("\n[INFO] Predicting data")
    predictions = model.forecast(len(test_df))
    # accuracy 
    print('\n[INFO] Model Accuracy:')
    results = forecast_accuracy(predictions, test_df)
    for metric, value in results.items():
        print(f'\t {metric}: {value}')


In [ ]:
%%writefile train_arrays.py

import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import joblib
import argparse
import os
from sagemaker.serializers import NumpySerializer



def forecast_accuracy(forecast, actual):
    #forecast, actual = np.asarray(forecast), np.asarray(actual)
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))*100  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return({'mape(%)':mape, 'mae': mae, 'rmse':rmse})

def _list_of_ints(arg):
    # for argparser
    return list(map(int, arg.split(',')))

if __name__=='__main__':
    parser = argparse.ArgumentParser()

    # where to write trained model artifacts
    parser.add_argument(
        "--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR")
    )

    # Train and Test Data
    parser.add_argument(
        "--train", type=str, default='daily_energy_train.csv'
    )
    parser.add_argument(
        "--test", type=str, default='daily_energy_test.csv'
        #default=os.path.join(
        #    os.environ.get("SM_CHANNEL_TRAIN"), "daily_energy_test.csv")
    )
    
    # Hyperparameters
    parser.add_argument(
        "--order", type=_list_of_ints, default=(3,0,0)
    )
    parser.add_argument(
        "--seasonal_order", type=_list_of_ints, default=(0,1,1,7)
    )

    args, _ = parser.parse_known_args()

    print("\n[INFO] Reading data\n")
    path = os.getcwd()
    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)
    
    print('train_df', train_df.head(3))
    
    # To array 
    train_df = np.array(train_df['PJME'])
    test_df = np.array(test_df['PJME'])
    
    print("\n[INFO] Training data with ARIMA")
    model = ARIMA(
        train_df,
        order=args.order,
        seasonal_order=args.seasonal_order
    )
    model = model.fit()

    model_path = os.path.join(args.model_dir, "model.joblib")
    print("\n[INFO] Saving model at", model_path)
    joblib.dump(model, model_path)


    print("\n[INFO] Predicting data")
    predictions = model.forecast(len(test_df))
    # accuracy 
    print('\n[INFO] Model Accuracy:')
    results = forecast_accuracy(predictions, test_df)
    for metric, value in results.items():
        print(f'\t {metric}: {value}')


In [ ]:
# for debug porpuses only, csv data can be passed as an dependency 
sklearn_estimator = SKLearn(
    entry_point='train_arrays.py',
    instance_type='ml.m5.4xlarge',
    framework_version='1.2-1',
    base_job_name="fidelidade-train-arrays",
    role=role,
    dependencies=['requirements.txt', 'daily_energy_train.csv', 'daily_energy_test.csv'], 
    hyperparameters={
        'order': "3,0,0",
        'seasonal_order': "0,1,1,7"
    },
)

In [ ]:
# training the model with S3 data 
sklearn_estimator.fit() 

# A) Use Estimator to Deploy and Predict

In [ ]:
endpoint_name = "fidelidade-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
print("EndpointName={}".format(endpoint_name))

predictor = sklearn_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.4xlarge',
    endpoint_name=endpoint_name,
    #predictor_cls=''
)

# B) Crete Model and Predictor from Artifacts

In [ ]:
sklearn_estimator.latest_training_job.wait(logs="None")
artifact = sm_boto3.describe_training_job(
    TrainingJobName=sklearn_estimator.latest_training_job.name
)["ModelArtifacts"]["S3ModelArtifacts"]

print("Model artifact persisted at " + artifact)

In [ ]:
model_name = "arima-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model = SKLearnModel(
    name = model_name,
    model_data=artifact,
    role=role,
    entry_point="train.py",
    framework_version='1.2-1',
    #predictor_cls=AnalysisClass     # Costumized class for Prediction (ex: serialization from json to pandas) 
)

In [ ]:
endpoint_name = "fidelidade-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name,
)

In [ ]:
predictor.__dict__

In [ ]:
test_df

In [ ]:
test_path = sagemaker_session.upload_data(test_df)

In [ ]:
response = predictor.predict(test_df)

In [ ]:
response = predictor.predict(np.array(test_df['PJME']))

In [ ]:
response = predictor.predict(np.array(test_df.loc[0, 'PJME']))

In [ ]:
response = predictor.predict(test_path)

In [ ]:
sm_boto3.delete_endpoint(EndpointName="fidelidade-2024-02-02-12-08-32")